In [33]:
import numpy as np
import pandas as pd

In [34]:
temp_df = pd.read_csv("IMDB Dataset.csv")

In [35]:
df = temp_df.iloc[:8000]

In [36]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [37]:
df.shape

(8000, 2)

In [38]:
df['review'][1]

'A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating every surface) are terribly well d

In [39]:
df['sentiment'].value_counts()

sentiment
positive    4003
negative    3997
Name: count, dtype: int64

In [40]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [41]:
df.duplicated().sum()

np.int64(10)

In [42]:
df.drop_duplicates(inplace=True)

In [43]:
df.duplicated().sum()

np.int64(0)

In [44]:
# Basic Preprocessing
# Remove tags
# lowercase
# remove stopwords

In [45]:
import re
def clean_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [46]:
df['review'] = df['review'].apply(clean_tags)

In [47]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
7995,"Wow, it's been years since I last saw this mov...",positive
7996,"This film is an impressionistic, poetic take o...",positive
7997,I never realized what a fabulous dancer Lana T...,positive
7998,There was nothing of value in the original mov...,negative


In [48]:
df['review'] = df['review'].apply(lambda x : x.lower())

In [49]:
from nltk.corpus import stopwords
sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

In [50]:
df

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production. filming technique...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there's family little boy (jake) thi...,negative
4,"petter mattei's ""love time money"" visually stu...",positive
...,...,...
7995,"wow, years since last saw movie. watching 2008...",positive
7996,"film impressionistic, poetic take immigrant ex...",positive
7997,never realized fabulous dancer lana turner saw...,positive
7998,"nothing value original movie, one even lamer. ...",negative


In [51]:
X = df.iloc[:,0:1]
y = df['sentiment']

In [52]:
X

,review
0,one reviewers mentioned watching 1 oz episode ...
1,wonderful little production. filming technique...
2,thought wonderful way spend time hot summer we...
3,basically there's family little boy (jake) thi...
4,"petter mattei's ""love time money"" visually stu..."
...,...
7995,"wow, years since last saw movie. watching 2008..."
7996,"film impressionistic, poetic take immigrant ex..."
7997,never realized fabulous dancer lana turner saw...
7998,"nothing value original movie, one even lamer. ..."


In [53]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [54]:
y

array([1, 1, 1, ..., 1, 0, 0], shape=(7990,))

In [55]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [56]:
X_train.shape

(6392, 1)

In [57]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer

In [58]:
cv = CountVectorizer()

In [59]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

In [60]:
X_train_bow.shape

(6392, 43653)

In [61]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

gnb.fit(X_train_bow,y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)number of training samples observed in each class.","ndarray[float64](2,)","[3200.,3192.]"
"class_prior_ class_prior_: ndarray of shape (n_classes,)probability of each class.","ndarray[float64](2,)","[0.5,0.5]"
"classes_ classes_: ndarray of shape (n_classes,)class labels known to the classifier.","ndarray[int64](2,)","[0,1]"
epsilon_ epsilon_: floatabsolute additive value to variances.,float64,5.233e-09
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,43653
"theta_ theta_: ndarray of shape (n_classes, n_features)mean of each feature per class.","ndarray[float64](2, 43653)","[[0. ,0.02,0. ,...,0. ,0. ,0. ], [0. ,0.01,0. ,...,0. ,0. ,0. ]]"
"var_ var_: ndarray of shape (n_classes, n_features)Variance of each feature per class... versionadded:: 1.0","ndarray[float64](2, 43653)","[[0. ,0.03,0. ,...,0. ,0. ,0. ], [0. ,0.01,0. ,...,0. ,0. ,0. ]]"


In [62]:
y_pred = gnb.predict(X_test_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(y_test, y_pred)

0.6389236545682102

In [63]:
confusion_matrix(y_test, y_pred)

array([[556, 235],
       [342, 465]])

In [64]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.844180225281602

In [65]:
cv = CountVectorizer(max_features=3000)

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8429286608260326

In [66]:
cv = CountVectorizer(ngram_range=(1,2),max_features=5000)

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8479349186483104

# Using TF-IDF

In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [68]:
tfidf = TfidfVectorizer()

In [69]:
X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()
X_test_tfidf = tfidf.transform(X_test['review']).toarray()

In [70]:

rf = RandomForestClassifier()

rf.fit(X_train_tfidf,y_train)
y_pred = rf.predict(X_test_tfidf)

accuracy_score(y_test,y_pred)

0.8448060075093867

# Using Word2Vec

In [74]:
import numpy as np
import pandas as pd

In [75]:
temp_df = pd.read_csv("IMDB Dataset.csv")

In [76]:
df = temp_df.iloc[:8000]

In [77]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [78]:
df.shape

(8000, 2)

In [80]:
df.drop_duplicates(inplace=True)

In [81]:
df.shape

(7990, 2)

In [82]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [83]:
df['review'] = df['review'].apply(remove_tags)

In [84]:
df['review'] = df['review'].apply(lambda x:x.lower())

In [85]:
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

In [86]:
df['review']

0       one reviewers mentioned watching 1 oz episode ...
1       wonderful little production. filming technique...
2       thought wonderful way spend time hot summer we...
3       basically there's family little boy (jake) thi...
4       petter mattei's "love time money" visually stu...
                              ...                        
7995    wow, years since last saw movie. watching 2008...
7996    film impressionistic, poetic take immigrant ex...
7997    never realized fabulous dancer lana turner saw...
7998    nothing value original movie, one even lamer. ...
7999    refused watch originally aired, treasuring mem...
Name: review, Length: 7990, dtype: str

In [87]:
import gensim

In [88]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [89]:
story = []

for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [90]:
story

[['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'hooked'],
 ['right',
  'exactly',
  'happened',
  'me',
  'the',
  'first',
  'thing',
  'struck',
  'oz',
  'brutality',
  'unflinching',
  'scenes',
  'violence',
  'set',
  'right',
  'word',
  'go'],
 ['trust', 'me', 'show', 'faint', 'hearted', 'timid'],
 ['show', 'pulls', 'punches', 'regards', 'drugs', 'sex', 'violence'],
 ['hardcore',
  'classic',
  'use',
  'word',
  'it',
  'called',
  'oz',
  'nickname',
  'given',
  'oswald',
  'maximum',
  'security',
  'state',
  'penitentary'],
 ['focuses',
  'mainly',
  'emerald',
  'city',
  'experimental',
  'section',
  'prison',
  'cells',
  'glass',
  'fronts',
  'face',
  'inwards',
  'privacy',
  'high',
  'agenda'],
 ['em',
  'city',
  'home',
  'many',
  'aryans',
  'muslims',
  'gangstas',
  'latinos',
  'christians',
  'italians',
  'irish',
  'more',
  'so',
  'scuffles',
  'death',
  'stares',
  'dodgy',
  'dealings',
  'shady',
  'agreements',
  'never',
  'far

In [91]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [92]:
model.build_vocab(story)

In [93]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(4687138, 4969820)

In [94]:
len(model.wv.index_to_key)

28807

In [96]:
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [97]:
document_vector(df['review'].values[0])

array([-0.288617  ,  0.43491292,  0.12170666,  0.03002449,  0.12816593,
       -0.64225364,  0.19827904,  0.82471794, -0.38716212, -0.3890356 ,
       -0.1754777 , -0.5055675 , -0.08177347,  0.2920156 ,  0.31840113,
       -0.44139764,  0.11156912, -0.35340318, -0.0839893 , -0.69056207,
        0.08322953,  0.12694664,  0.10316152, -0.22491309, -0.00847708,
       -0.092457  , -0.23155908, -0.17931001, -0.45725107, -0.16255133,
        0.26413223,  0.06448447, -0.09798689, -0.27590322, -0.10935629,
        0.48201045,  0.01638818, -0.13341224, -0.2373221 , -0.6497658 ,
        0.09578875, -0.4880918 , -0.06670222, -0.26911262,  0.20720069,
       -0.12956728, -0.31628793, -0.05283045,  0.17511682,  0.35446545,
        0.15423666, -0.3074238 , -0.32092074, -0.16017185, -0.25131926,
       -0.0520925 ,  0.22451535,  0.07559993, -0.39646617,  0.22211514,
       -0.05321201,  0.07986412,  0.0315989 , -0.02438958, -0.41562614,
        0.44325274,  0.05704368,  0.12804419, -0.5389757 ,  0.33

In [98]:
from tqdm import tqdm

In [99]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))


00%|██████████████████████████████████████████████████████████████████████████████| 7990/7990 [20:32<00:00,  6.48it/s]

In [100]:
X = np.array(X)

In [101]:
X[0]

array([-0.288617  ,  0.43491292,  0.12170666,  0.03002449,  0.12816593,
       -0.64225364,  0.19827904,  0.82471794, -0.38716212, -0.3890356 ,
       -0.1754777 , -0.5055675 , -0.08177347,  0.2920156 ,  0.31840113,
       -0.44139764,  0.11156912, -0.35340318, -0.0839893 , -0.69056207,
        0.08322953,  0.12694664,  0.10316152, -0.22491309, -0.00847708,
       -0.092457  , -0.23155908, -0.17931001, -0.45725107, -0.16255133,
        0.26413223,  0.06448447, -0.09798689, -0.27590322, -0.10935629,
        0.48201045,  0.01638818, -0.13341224, -0.2373221 , -0.6497658 ,
        0.09578875, -0.4880918 , -0.06670222, -0.26911262,  0.20720069,
       -0.12956728, -0.31628793, -0.05283045,  0.17511682,  0.35446545,
        0.15423666, -0.3074238 , -0.32092074, -0.16017185, -0.25131926,
       -0.0520925 ,  0.22451535,  0.07559993, -0.39646617,  0.22211514,
       -0.05321201,  0.07986412,  0.0315989 , -0.02438958, -0.41562614,
        0.44325274,  0.05704368,  0.12804419, -0.5389757 ,  0.33

In [102]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [103]:
y

array([1, 1, 1, ..., 1, 0, 0], shape=(7990,))

In [104]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [105]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [106]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.7571964956195244